## Write a DSPy Program That Utilizes Tools in MCP Server
Let's build an agent which utilizes the MCP tools in our server to assist users.

In [8]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
import dspy
from os  import getenv
from dotenv import load_dotenv
import json

Load environment variables and configuration file

In [9]:
load_dotenv(dotenv_path="../.env")
SERVER_HOST = getenv("SERVER_HOST", "localhost")

# Load config from file
with open("../claude_desktop_config.json", "r") as f:
    config = json.load(f)

Adding observability

In [ ]:
# import mlflow

# # Set the MLflow Tracking URI
# mlflow.set_tracking_uri(f"http://{SERVER_HOST}:5000")

# # Set the experiment (creates it if it doesn't exist)
# mlflow.set_experiment("DSPy Jira Agent Test")

# # Optional: Start a run
# with mlflow.start_run(run_name="test-run"):
#     mlflow.log_param("param1", 10)
#     mlflow.log_metric("accuracy", 0.85)

Extract config for a specific server

In [10]:
mcp_config = config["mcpServers"]["mcp-atlassian"]

# Build the server parameters
server_params = StdioServerParameters(
    command=mcp_config["command"],
    args=mcp_config["args"],
    env=mcp_config["env"]
)

### Gather and List Tools from MCP Servers
We first need to gather all available tools from the MCP server and make them usable by DSPy. DSPy provides an API `dspy.Tool` as the standard tool interface. Let's convert all the MCP tools to dspy.Tool.

We need to create an MCP client instance to communicate with the MCP server, fetch all available tools, and convert them to `dspy.Tool` using the static method `from_mcp_tool`:

In [26]:
available_tools = []

async with stdio_client(server_params) as (read, write):
    async with ClientSession(read, write) as session:
        # Initialize the connection
        await session.initialize()
        # List available tools
        tools = await session.list_tools()

        # Convert MCP tools to DSPy tools
        dspy_tools = []
        for tool in tools.tools:
            dspy_tools.append(dspy.Tool.from_mcp_tool(session, tool))

        # Extract tool names
        available_tools = sorted(tool.name for tool in dspy_tools)

Print the number of tools and their arguments

In [32]:
import math

# Parameters
num_columns = 4
column_width = max(len(name) for name in available_tools) + 4
num_rows = math.ceil(len(available_tools) / num_columns)

# Organize into columns
columns = [available_tools[i * num_rows:(i + 1) * num_rows] for i in range(num_columns)]

# Pad shorter columns
for col in columns:
    while len(col) < num_rows:
        col.append("")

# Print output
print("Number of tools:", len(available_tools))
for row in zip(*columns):
    print("".join(name.ljust(column_width) for name in row))

Number of tools: 27
jira_add_comment               jira_delete_issue              jira_get_project_versions      jira_remove_issue_link         
jira_add_worklog               jira_download_attachments      jira_get_sprint_issues         jira_search                    
jira_batch_create_issues       jira_get_agile_boards          jira_get_sprints_from_board    jira_search_fields             
jira_batch_get_changelogs      jira_get_board_issues          jira_get_transitions           jira_transition_issue          
jira_create_issue              jira_get_issue                 jira_get_user_profile          jira_update_issue              
jira_create_issue_link         jira_get_link_types            jira_get_worklog               jira_update_sprint             
jira_create_sprint             jira_get_project_issues        jira_link_to_epic                                             


## Build a DSPy Agent to Handle User Requests
Now we will use `dspy.ReAct` to build the agent for handling User requests. `ReAct` stands for "reasoning and acting," which asks the LLM to decide whether to call a tool or wrap up the process. If a tool is required, the LLM takes responsibility for deciding which tool to call and providing the appropriate arguments.

As usual, we need to create a `dspy.Signature` to define the input and output of our agent:

In [28]:
import dspy

class DSPyJiraUserService(dspy.Signature):
    """
    You are a JIRA user service agent. You are given a list of tools to handle user requests.
    
    You should decide the right tool to use in order to fulfill users' requests.
    """

    user_request: str = dspy.InputField()
    process_result: str = dspy.OutputField(
        desc=(
            "Message that summarizes the process result, and the information users need, "
            "e.g., the issue_key if it's a JIRA issue request."
        )
    )

And choose an LM for our agent:

In [29]:
dspy.configure(
    lm=dspy.LM(
        model="ollama_chat/qwen3:14b",  # e.g., "qwen3:8b", "qwen3:1.7b", "qwen3:14b"
        api_base=f"http://{SERVER_HOST}:11434",
        api_key=""  # Dummy key, Ollama doesn't require authentication
    )
)

# api_key = ""
# dspy.configure(lm=dspy.LM(
#     'gemini/gemini-2.5-flash',
#     api_key=api_key
# ))

# api_key=""
# dspy.configure(
#     lm=dspy.LM(
#         model=f"openai/gpt-4o",  # e.g., "gpt-3.5-turbo", "gpt-4"
#         api_key="",
#         cache=False
#     )
# )

Let's create an MCP client that takes a request

In [30]:
async def run(user_request):
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            # Initialize the connection
            await session.initialize()
            # List available tools
            tools = await session.list_tools()

            # Convert MCP tools to DSPy tools
            dspy_tools = []
            for tool in tools.tools:
                dspy_tools.append(dspy.Tool.from_mcp_tool(session, tool))

            # # Gather only the tools we want to use
            # tools_to_use = ["jira_get_issue", "jira_search_issues", "jira_create_issue", "jira_search"]
            # for tool in tools.tools:
            #     if tool.name in tools_to_use:
            #         dspy_tools.append(dspy.Tool.from_mcp_tool(session, tool))

            # Create the agent
            react = dspy.ReAct(DSPyJiraUserService, tools=dspy_tools)

            result = await react.acall(user_request=user_request)
            print(result)

Let's test our Agent

In [31]:
# await run("Create a story in project LLM titled 'Enable OAuth login' with description 'Add Google login support via OAuth2.'",)
await run("Do i have any project available in JIRA?")

Prediction(
    trajectory={'thought_0': 'The user is asking if they have any projects available in JIRA. To determine this, we need to check if there are projects they have access to or are associated with. However, the available tools do not directly support listing all projects. Instead, we can search for issues assigned to the user, which would imply they are part of projects. Alternatively, if the user is asking about projects they can access, we might need to use a different approach, but the available tools do not support that directly. The most relevant tool here is `jira_search` to find issues related to the user, which would indicate project involvement.', 'tool_name_0': 'jira_search', 'tool_args_0': {'jql': 'assignee = currentUser() AND project IS NOT EMPTY', 'maxResults': 100}, 'observation_0': 'Execution error in jira_search: \nTraceback (most recent call last):\n  File "/home/facko/workplace/omniopenverse/llm_works/agentic/.venv/lib/python3.13/site-packages/dspy/predict/r